<a href="https://colab.research.google.com/github/tharun-7733/Crop_recommendation/blob/master/Crop_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
atharvaingle_crop_recommendation_dataset_path = kagglehub.dataset_download('atharvaingle/crop-recommendation-dataset')

print('Data source import complete.')

Using Colab cache for faster access to the 'crop-recommendation-dataset' dataset.
Data source import complete.


In [ ]:
import os

# Locate the CSV file within the downloaded path
csv_path = None
for dirname, _, filenames in os.walk(atharvaingle_crop_recommendation_dataset_path):
    for filename in filenames:
        if filename.endswith('.csv'):
            csv_path = os.path.join(dirname, filename)
            break

# Load the dataset
if csv_path:
    df = pd.read_csv(csv_path)
    print(f'Dataset loaded successfully from: {csv_path}')
    display(df.head())
else:
    print('CSV file not found in the dataset path.')

Dataset loaded successfully from: /kaggle/input/crop-recommendation-dataset/Crop_recommendation.csv


,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [ ]:
df.describe()

,N,P,K,temperature,humidity,ph,rainfall
count,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000,2200.000000
mean,50.551818,53.362727,48.149091,25.616244,71.481779,6.469480,103.463655
std,36.917334,32.985883,50.647931,5.063749,22.263812,0.773938,54.958389
min,0.000000,5.000000,5.000000,8.825675,14.258040,3.504752,20.211267
25%,21.000000,28.000000,20.000000,22.769375,60.261953,5.971693,64.551686
50%,37.000000,51.000000,32.000000,25.598693,80.473146,6.425045,94.867624
75%,84.250000,68.000000,49.000000,28.561654,89.948771,6.923643,124.267508
max,140.000000,145.000000,205.000000,43.675493,99.981876,9.935091,298.560117


In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   N            2200 non-null   int64  
 1   P            2200 non-null   int64  
 2   K            2200 non-null   int64  
 3   temperature  2200 non-null   float64
 4   humidity     2200 non-null   float64
 5   ph           2200 non-null   float64
 6   rainfall     2200 non-null   float64
 7   label        2200 non-null   object 
dtypes: float64(4), int64(3), object(1)
memory usage: 137.6+ KB


In [ ]:
df.isnull().sum()

,0
N,0
P,0
K,0
temperature,0
humidity,0
ph,0
rainfall,0
label,0


In [ ]:
df.drop_duplicates(df, inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler

# Initialize encoder and scaler
le = LabelEncoder()
scaler = StandardScaler()

# Features and Target
X = df.drop('label', axis=1)
y = df['label']

# Scale the numerical features
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

# Encode the target labels
y_encoded = le.fit_transform(y)

print('Feature Engineering complete:')
print(f'Encoded Labels: {dict(zip(le.classes_, le.transform(le.classes_)))}')
display(X_scaled_df.head())

Feature Engineering complete:
Encoded Labels: {'apple': np.int64(0), 'banana': np.int64(1), 'blackgram': np.int64(2), 'chickpea': np.int64(3), 'coconut': np.int64(4), 'coffee': np.int64(5), 'cotton': np.int64(6), 'grapes': np.int64(7), 'jute': np.int64(8), 'kidneybeans': np.int64(9), 'lentil': np.int64(10), 'maize': np.int64(11), 'mango': np.int64(12), 'mothbeans': np.int64(13), 'mungbean': np.int64(14), 'muskmelon': np.int64(15), 'orange': np.int64(16), 'papaya': np.int64(17), 'pigeonpeas': np.int64(18), 'pomegranate': np.int64(19), 'rice': np.int64(20), 'watermelon': np.int64(21)}


,N,P,K,temperature,humidity,ph,rainfall
0,1.068797,-0.344551,-0.101688,-0.935587,0.472666,0.043302,1.810361
1,0.933329,0.140616,-0.141185,-0.759646,0.397051,0.734873,2.242058
2,0.255986,0.049647,-0.081939,-0.515898,0.486954,1.771510,2.921066
3,0.635298,-0.556811,-0.160933,0.172807,0.389805,0.660308,2.537048
4,0.743673,-0.344551,-0.121436,-1.083647,0.454792,1.497868,2.898373


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
import numpy as np

# Define models
models = {
    'Random Forest': RandomForestClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42),
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42)
}

# Evaluate models using 5-fold cross-validation
results = {}
for name, model_obj in models.items():
    cv_scores = cross_val_score(model_obj, X, y, cv=5)
    results[name] = cv_scores.mean()
    print(f'{name} CV Accuracy: {cv_scores.mean():.2%}')

# Select the best model
best_model_name = max(results, key=results.get)
print(f'\nBest Model identified: {best_model_name}')

# Final training on the training split
best_model = models[best_model_name]
best_model.fit(X_train, y_train)

# Evaluation on the test set
y_pred = best_model.predict(X_test)
print(f'Final Test Accuracy with {best_model_name}: {best_model.score(X_test, y_test):.2%}')

Random Forest CV Accuracy: 99.45%
SVM CV Accuracy: 97.82%


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

Logistic Regression CV Accuracy: 97.09%

Best Model identified: Random Forest
Final Test Accuracy with Random Forest: 99.32%


In [ ]:
import time
import pandas as pd

train_eval_results = []

for name, model_obj in models.items():
    # Measure training time
    start_time = time.time()
    model_obj.fit(X_train, y_train)
    end_time = time.time()

    train_acc = model_obj.score(X_train, y_train)
    test_acc = model_obj.score(X_test, y_test)

    train_eval_results.append({
        'Model': name,
        'Training Accuracy': f'{train_acc:.2%}',
        'Test Accuracy': f'{test_acc:.2%}',
        'Training Time (s)': f'{end_time - start_time:.4f}'
    })

# Display the evaluation summary
eval_df = pd.DataFrame(train_eval_results)
print('--- Training vs Test Performance Comparison ---')
display(eval_df)

--- Training vs Test Performance Comparison ---


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


,Model,Training Accuracy,Test Accuracy,Training Time (s)
0,Random Forest,100.00%,99.32%,0.6793
1,SVM,98.12%,96.14%,0.3772
2,Logistic Regression,98.12%,95.23%,2.5555


In [ ]:
def recommend_crop(N, P, K, temperature, humidity, ph, rainfall):
    """
    Function to predict the best crop based on soil and weather inputs.
    """
    # Create a dataframe for the input to match feature names
    features = pd.DataFrame([[N, P, K, temperature, humidity, ph, rainfall]],
                            columns=['N', 'P', 'K', 'temperature', 'humidity', 'ph', 'rainfall'])

    prediction = best_model.predict(features)
    return prediction[0]

# Example usage with a sample data point from the dataset
example_crop = recommend_crop(90, 42, 43, 20.87, 82.00, 6.50, 202.93)
print(f'Recommended Crop for sample conditions: {example_crop}')

Recommended Crop for sample conditions: rice


In [ ]:
import pickle

# Save the best model to a file
model_filename = 'crop_recommendation_model.pkl'
with open(model_filename, 'wb') as file:
    pickle.dump(best_model, file)

print(f'Model saved successfully as {model_filename}')

Model saved successfully as crop_recommendation_model.pkl


In [ ]:
import pickle

# Save the scaler object to a file
scaler_filename = 'scaler.pkl'
with open(scaler_filename, 'wb') as file:
    pickle.dump(scaler, file)

print(f'Scaler saved successfully as {scaler_filename}')

Scaler saved successfully as scaler.pkl
